In [1]:
%cd ..

/home/hsc/Projects/personal/disertation


In [2]:
from pathlib import Path


DATA_DIRPATH = Path("./data")
PROJECT_DIRPATH = Path("./iterations")
TRAIN_DATA_DIRPATH = DATA_DIRPATH / "train"
TEST_DATA_DIRPATH = DATA_DIRPATH / "test"
TRAIN_CSV_FILEPATH = DATA_DIRPATH / "train_labels.csv"
THRESH_CSV_FILEPATH = DATA_DIRPATH / "train_thresholds.csv"

In [3]:
import torch

DEVICE = torch.device("cuda:0")

In [4]:
import dataclasses
import os

import numpy as np
import pandas as pd

from app.imc2025.prediction import load_from_test, load_from_train



# Set is_train=True to run the notebook on the training data.
# Set is_train=False if submitting an entry to the competition (test data is hidden, and different from what you see on the "test" folder).
is_train = True
data_dir = DATA_DIRPATH
workdir = PROJECT_DIRPATH
os.makedirs(workdir, exist_ok=True)
samples = None
if is_train:
    samples = load_from_train(DATA_DIRPATH)
else:
    samples = load_from_test(DATA_DIRPATH)

for dataset in samples:
    print(f'Dataset "{dataset}" -> num_images={len(samples[dataset])}')

Dataset "imc2023_haiper" -> num_images=54
Dataset "imc2023_heritage" -> num_images=209
Dataset "imc2023_theather_imc2024_church" -> num_images=76
Dataset "imc2024_dioscuri_baalshamin" -> num_images=138
Dataset "imc2024_lizard_pond" -> num_images=214
Dataset "pt_brandenburg_british_buckingham" -> num_images=225
Dataset "pt_piazzasanmarco_grandplace" -> num_images=168
Dataset "pt_sacrecoeur_trevi_tajmahal" -> num_images=225
Dataset "pt_stpeters_stpauls" -> num_images=200
Dataset "amy_gardens" -> num_images=200
Dataset "fbk_vineyard" -> num_images=163
Dataset "ETs" -> num_images=22
Dataset "stairs" -> num_images=51


In [5]:
from mts.pipeline.pipeline.imc2025 import IMC2025Pipeline

/home/hsc/Projects/personal/disertation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from __future__ import annotations
from mts.core.types import PathLike



In [7]:
from hydra.utils import instantiate
from omegaconf import OmegaConf

In [8]:
cfg = OmegaConf.load("config/pipeline/imc2025/0001.yaml")

In [ ]:
from mts.pipeline.step.base import BasePipelineStep



In [11]:
from mts.pipeline.repository.inmemeory import ImageRepository

predictions = samples["imc2023_haiper"]
image_repository = ImageRepository()

for pred in predictions:
    image_repository.add_image(pred.image_filepath)

In [12]:
last_project_iteration = Project.from_next_iteration("iterations")

In [13]:
state = {
    "images_dir": ".",
    "colmap_dirpath": last_project_iteration.iteration_dirpath,
}

In [10]:
from mts.core.types import StateType
from mts.pipeline.repository.inmemeory import ImageRepository


def create_repository(dataset_name: str) -> ImageRepository:
    image_repository = ImageRepository()
    image_repository.add_repository_metadata(dataset_name=dataset_name)
    return image_repository


def create_pipeline(
    pipeline_config_filepath: PathLike,
    image_repository: ImageRepository,
    dataset_name: str,
) -> list[BasePipelineStep]:
    pipeline_steps = from_hydra_config(pipeline_config_filepath)
    return pipeline_steps


def create_pipeline_state(
    imc2025_pipeline: IMC2025Pipeline,
    image_repository: ImageRepository,
    dataset_name: str,
) -> StateType:
    dataset_dirpath = Path(imc2025_pipeline.project_dirpath) / dataset_name
    dataset_dirpath.mkdir(exist_ok=True)
    state = {
        "images_dir": ".",
        "colmap_dirpath": dataset_dirpath,
    }
    return state

In [11]:
pipeline_steps = from_hydra_config("config/pipeline/imc2025/0001.yaml")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loaded LightGlue model


In [12]:
from tqdm.auto import tqdm

image_repository = create_repository("imc2023_haiper")
for sample in tqdm(samples["imc2023_haiper"]):
    image_repository.add_image(sample.image_filepath)

100%|██████████| 54/54 [00:00<00:00, 76543.57it/s]


In [13]:
from functools import partial


last_project_iteration = Project.from_next_iteration("iterations")
imc2025_pipeline = IMC2025Pipeline(
    last_project_iteration.iteration_dirpath,
    samples,
    create_repository,
    partial(create_pipeline, "config/pipeline/imc2025/0001.yaml"),
    create_pipeline_state=create_pipeline_state,
)

In [14]:
dataset_name = "imc2023_haiper"
dataset_names = [ "imc2023_haiper", "pt_brandenburg_british_buckingham"]

In [15]:
imc2025_pipeline.run(dataset_names)

Loaded LightGlue model


Add matches: 100%|██████████| 1431/1431 [00:00<00:00, 358309.89it/s]
I20260102 16:41:42.241785 139854369240640 misc.cc:44] 
Feature matching & geometric verification
I20260102 16:41:42.243148 139852632815168 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244204 139852624422464 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244335 139852616029760 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244355 139852607637056 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244431 139852515370560 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244566 139852506977856 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.244469 139852523763264 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.245551 139852121110080 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.245638 139852490192448 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:41:42.24611

Loaded LightGlue model


Add matches: 100%|██████████| 25200/25200 [00:00<00:00, 147263.29it/s]
I20260102 16:55:23.653482 139851315803712 misc.cc:44] 
Feature matching & geometric verification
I20260102 16:55:23.653879 139852070753856 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.653909 139852079146560 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.653938 139852087539264 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.653949 139854369240640 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.654822 139852632815168 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.654873 139852624422464 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.654905 139852616029760 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.654932 139852607637056 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.655950 139852523763264 sift.cc:1452] Creating SIFT CPU feature matcher
I20260102 16:55:23.655

In [ ]:
last_project_iteration = Project.from_next_iteration("iterations")
dataset_dirpath = Path(last_project_iteration.iteration_dirpath) / dataset_name
dataset_dirpath.mkdir(exist_ok=True)
state = {
    "images_dir": ".",
    "colmap_dirpath": dataset_dirpath,
}

In [23]:
submission_filepath = last_project_iteration.iteration_dirpath / "submission.csv"

In [24]:
to_df(imc2025_pipeline.samples).to_csv(
    submission_filepath,
    sep=",",
    index=False,
)

In [ ]:
last_project_iteration = Pro

In [28]:
from mts.helpers.imc import metric


final_score, dataset_scores, scene_clusterness_score = metric.score(
    gt_csv='data/train_labels.csv',
    user_csv=submission_filepath,
    thresholds_csv='data/train_thresholds.csv',
    mask_csv=None if is_train else os.path.join(data_dir, 'mask.csv'),
    inl_cf=0,
    strict_cf=-1,
    verbose=True,
)

imc2023_haiper: score=52.72% (mAA=41.11%, clusterness=73.47%)
imc2023_heritage: score=0.00% (mAA=0.00%, clusterness=100.00%)
imc2023_theather_imc2024_church: score=0.00% (mAA=0.00%, clusterness=100.00%)
imc2024_dioscuri_baalshamin: score=0.00% (mAA=0.00%, clusterness=100.00%)
imc2024_lizard_pond: score=0.00% (mAA=0.00%, clusterness=100.00%)
pt_brandenburg_british_buckingham: score=11.01% (mAA=6.60%, clusterness=33.33%)
pt_piazzasanmarco_grandplace: score=0.00% (mAA=0.00%, clusterness=100.00%)
pt_sacrecoeur_trevi_tajmahal: score=0.00% (mAA=0.00%, clusterness=100.00%)
pt_stpeters_stpauls: score=0.00% (mAA=0.00%, clusterness=100.00%)
amy_gardens: score=0.00% (mAA=0.00%, clusterness=100.00%)
fbk_vineyard: score=0.00% (mAA=0.00%, clusterness=100.00%)
ETs: score=0.00% (mAA=0.00%, clusterness=100.00%)
stairs: score=0.00% (mAA=0.00%, clusterness=100.00%)
Average over all datasets: score=4.90% (mAA=3.67%, clusterness=92.83%)


TypeError: 'numpy.float64' object is not iterable

In [27]:
final_score

(np.float64(4.90274219251409), np.float64(4.90274219251409), np.float64(0.0))

In [26]:
dataset_scores

({'imc2023_haiper': np.float64(52.72115574905997),
  'imc2023_heritage': np.float64(0.0),
  'imc2023_theather_imc2024_church': np.float64(0.0),
  'imc2024_dioscuri_baalshamin': np.float64(0.0),
  'imc2024_lizard_pond': np.float64(0.0),
  'pt_brandenburg_british_buckingham': np.float64(11.01449275362319),
  'pt_piazzasanmarco_grandplace': np.float64(0.0),
  'pt_sacrecoeur_trevi_tajmahal': np.float64(0.0),
  'pt_stpeters_stpauls': np.float64(0.0),
  'amy_gardens': np.float64(0.0),
  'fbk_vineyard': np.float64(0.0),
  'ETs': np.float64(0.0),
  'stairs': np.float64(0.0)},
 None,
 None)

In [ ]:
from mts.pipeline.step.base import run_pipeline


run_pipeline(
    steps=pipeline_steps,
    image_repository=image_repository,
    input=None,
    state=state,
)

Match the keypoints and descriptors: 100%|██████████| 1431/1431 [00:34<00:00, 41.23it/s]


In [ ]:
from typing import Any
import logging

from mts.core.types import StateType
from mts.pipeline.step.base import run_pipeline

LOGGER = logging.getLogger(__name__)


class ReconstructionPipeline(BasePipelineStep):
    def __init__(
        self,
        repository: ImageRepository,
        steps: list[BasePipelineStep],
    ) -> None:
        super().__init__()
        self.steps = steps
        self.repository = repository

    def set_repository(self, repository: ImageRepository) -> None:
        LOGGER.info("Setup repository for the whole pipeline")
        for step in self.steps:
            step.set_image_repository(repository)

    def run(
        self,
        *,
        input: Any | None,
        state: StateType | None = None,
    ) -> Any:
        self._set_up()
        result = run_pipeline(self.steps, input=input, state=state)
        return result

In [29]:
from hloc.utils import viz as img_viz

In [ ]:
st_idx, nd_idx = 42, 54
img_viz.plot_images(
    [image_repository.load_image(st_idx), image_repository.load_image(nd_idx)]
)
st_kp = image_repository.get_keypoints(st_idx)
nd_kp = image_repository.get_keypoints(nd_idx)

img_viz.plot_keypoints([st_kp, nd_kp])
matches = image_repository.get_matches(st_idx, nd_idx)
img_viz.plot_matches(st_kp[matches[:, st_idx]], nd_kp[matches[:, nd_idx]])

KeyError: 54